# Setup
Run this notebook ONCE before executing the pipeline for the first time.
Creates all required schemas and tables.

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
print(f"catalog_name   : {catalog_name}")

catalog_name   : workspace


In [0]:
for schema in ["final_project_raw", "final_project_bronze", "final_project_silver", "final_project_gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")
    print(f"Schema ready: {catalog_name}.{schema}")

Schema ready: workspace.final_project_raw
Schema ready: workspace.final_project_bronze
Schema ready: workspace.final_project_silver
Schema ready: workspace.final_project_gold


### Create Raw Zone Volume (run manually or via SQL)

If the volume does not yet exist, run the following in a SQL cell or Catalog UI:

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.final_project_raw.food_inspection

### Create Bronze Zone Volume (run manually or via SQL)

If the volume does not yet exist, run the following in a SQL cell or Catalog UI:

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.final_project_bronze.food_inspection

### Create Parent Metadata Table

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.final_project_bronze.pipeline_metadata_parent (
        table_name    STRING        NOT NULL,
        file_name     STRING,
        active_flag   STRING        NOT NULL,
        created_date  DATE,
        modified_date DATE
    )
    USING DELTA
""")
print("pipeline_metadata_parent ready.")

pipeline_metadata_parent ready.


### Seed Parent Metadata — Food Inspection Tables

In [0]:
from datetime import date

today = date.today()

chinook_tables = [
    ("Food_Inspections_Chicago", "Food_Inspections_Chicago.csv"),
    ("Food_Inspections_Dallas", "Food_Inspections_Dallas.csv")
]

existing = spark.table(f"{catalog_name}.final_project_bronze.pipeline_metadata_parent")
existing_names = [r.table_name for r in existing.select("table_name").collect()]

rows_to_insert = [
    (t, f, "Y", today, today)
    for t, f in chinook_tables
    if t not in existing_names
]

if rows_to_insert:
    from pyspark.sql.types import StructType, StructField, StringType, DateType
    schema = StructType([
        StructField("table_name",    StringType(), True),
        StructField("file_name",     StringType(), True),
        StructField("active_flag",   StringType(), True),
        StructField("created_date",  DateType(),   True),
        StructField("modified_date", DateType(),   True),
    ])
    seed_df = spark.createDataFrame(rows_to_insert, schema)
    seed_df.write.format("delta").mode("append").saveAsTable(
        f"{catalog_name}.final_project_bronze.pipeline_metadata_parent"
    )
    print(f"Inserted {len(rows_to_insert)} rows into pipeline_metadata_parent.")
else:
    print("Parent metadata already seeded — nothing inserted.")

display(spark.table(f"{catalog_name}.final_project_bronze.pipeline_metadata_parent"))

Inserted 2 rows into pipeline_metadata_parent.


table_name,file_name,active_flag,created_date,modified_date
Food_Inspections_Chicago,Food_Inspections_Chicago.csv,Y,2026-04-08,2026-04-08
Food_Inspections_Dallas,Food_Inspections_Dallas.csv,Y,2026-04-08,2026-04-08


### Create Child Execution Metrics Table

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.final_project_bronze.pipeline_metadata_child (
        table_name          STRING,
        execution_time      TIMESTAMP,
        status              STRING,
        source_row_count    INT,
        target_row_count    INT,
        file_location       STRING,
        created_date        DATE
    )
    USING DELTA
""")
print("pipeline_metadata_child ready.")

pipeline_metadata_child ready.


### Silver tables for DQX Logging and Quarantine

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.final_project_silver.dqx_execution_log (
        table_name      STRING,
        run_timestamp   TIMESTAMP,
        passed_records  BIGINT,
        failed_records  BIGINT,
        created_date    DATE
    ) USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.final_project_silver.chicago_quarantine (
        table_name      STRING,
        run_timestamp   TIMESTAMP,
        _error_details  STRING,
        _row_data       STRING
    ) USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.final_project_silver.dallas_quarantine (
        table_name      STRING,
        run_timestamp   TIMESTAMP,
        _error_details  STRING,
        _row_data       STRING
    ) USING DELTA
""")

DataFrame[]